# The Attention Mechanism

### Main Steps
- Vocabulary and Tokenization
- Context Windows
- Input Embeddings
- Positional Encoding
- Update of Embeddings via Attention
- Next Word Prediction & Training

```{figure} ../../images/copyrighted/transformer_diagram.png
:alt: transformer
:height: 500px
:align: center

Vaswani, Ashish, et al. "Attention is all you need." Advances in neural information processing systems 30 (2017).
```

### Multi-head Masked Self-Attention

- **Attention**: Calculates the importance of each token relative to others, helping the model understand context. Tokens/embeddings **attend to** other tokens/embeddings.
- **Self**: Each token in the sequence pays attention to all other tokens, as opposed to **cross-attention**, which is used in tasks like language translation.
- **Masked**: Prevents access to future tokens since we learn by next-word prediction.
- **Multi-Head**: Multiple attention mechanisms work in parallel to capture different aspects of the input sequence simultaneously.

For example, consider the text: 
> "In August, water consumption in Australia is usually ___"
- The model needs to predict what comes next. Options could include "higher," "lower," or something else.
- In decoder transformers, only the "deepest" (i.e., most refined through layers) embedding of the final word is used to predict the next word. For example, the embedding of "usually" is refined to predict the next word accurately.

The model should recognize the context:
- Subject (What?),
- Time (When?),
- Place (Where?).

Without attending to "Australia," the model might incorrectly predict "higher."


### Updating Embeddings
To update embeddings, the **Attention** mechanism uses a **QUERY**, **KEY**, and **VALUE** triplet.

```{figure} ../../images/update_embeddings.png
:alt: update_embeddings
:height: 350px
:align: center

Embeddings Updating in Attention Mechanism
```

- **Query (Q)**: A set criteria or the question each token asks about the others.
- **Key (K)**: Information each token provides in response to queries.
- **Value (V)**: Data used to update or transform other embeddings.
  
**Q**, **K**, and **V** are vectors computed from embeddings by multiplying with matrices of learnable parameters.


When updating the embeddings of a token, the model computes how much it has to attend to on every preceding token. How?
* It uses the dot product to compute the similarity between the token’s Query vector and the Key vectors of all other tokens. 
* This results in a set of attention weights.
* The attention weights are then used to compute a weighted sum of the Value vectors from all tokens to update the embedding.


### QUERY and KEY
Example text: "In August, water consumption in Australia is usually ___"

- **QUERY**: "Is there a location or place before me?"
- **KEY**: Response from tokens, e.g., "Yeah!"

```{figure} ../../images/query_computing.png
:alt: query_computing
:height: 350px
:align: center

Computing the Query in the Attention Mechanism
```

To calculate the query for the eigth token we use:

$$
W_Q \vec{E_8} = \vec{Q_8}
$$

And if we want to compute the key for the sixth token, we use:

$$
W_K \vec{E_6} = \vec{K_6}
$$

Where $W_Q$ contains the learnable parameters that are the same for all the tokens. Only initial embeddings are learned, the rest are computed.


### Estimating Relevance from QUERY and KEY
- Relevance is estimated by vector similarity (dot product).
- Dot product is transformed into a distribution summing to 1 with **Softmax Activation**.

Where the softmax activation function is:

$$
\sigma (z_i) \dfrac{e^{z_i}}{\sum^n_{j=1} e^{z_j}}
$$

**Softmax** creates attention weights to weight the Value vectors, i.e.

$$
Attention(Q,K,V) = softmax(\dfrac{QK^T}{\sqrt{d_k}}V)
$$

Where the "Value" represents the contribution of a given embedding to adjusting another embedding.

### Updating the embeddings
We finally update the embeddings with the corresponding attention weights (calculated using the softmax function),

$$ 
\vec{E^\prime_8} = E_8 + \vec{\Delta E_8}
$$

$$
= \vec{E_8} + \omega_0 \vec{V_0}+ \omega_0 \vec{V_0} + ... + + \omega_8 \vec{V_8}
$$


### Multi-head attention
* Multi-head attention implies adding multiple $\vec{\Delta E_8}$ on for each head; each head has its Q,K,V; GPT-3 had 96 heads

$$ 
\vec{E^\prime_8} = \vec{E_8} + \vec{\Delta E^8}^{(0)}  + \vec{\Delta E^8}^{(1)} + ... + \vec{\Delta E^8}^{(95)}
$$

* Each attention block is followed by an MLP adds nonlinearities to enhance the model’s expression power.
* There are 96 attention blocks in GPT-3
* After all this “deep”, very expensive, processing… how do we make the final prediction?


### Final Steps for Next-Word Prediction
- The final layer size matches the vocabulary (e.g., >50k for GPT-3).
- A **softmax function** returns a probability for each word in the vocabulary.
- During training, probability is used to compute loss, and during testing, it predicts the next word.


Also important to note is that training occurs in parallel for the learnable parameters, i.e.
Example sequence and labels to predict:
- "In ___" → Label: "August"
- "In August ___" → Label: ","
- Continue with "water," "consumption," "in," "Australia," "is," and "usually."

### Summary
- **Attention Mechanism**: Each token is represented by three vectors (Q, K, V).
  - **Query**: Criteria/questions each token asks about others in the sequence.
  - **Key**: Information each token provides in response.
  - **Value**: Data used for updating other token embeddings.
- Q, K, V are computed from embeddings with learned \( W_Q \), \( W_K \), \( W_V \).
- **Attention Weights**: Calculated by Query-Key similarity, applied to Value vectors for updates.
- **Multi-head Attention**: Allows the model to capture multiple input aspects simultaneously.
- **Stacked Attention Blocks**: Refine representations progressively.
- **MLP Layers**: Add nonlinearity, helping learn complex feature interactions.
- The transformer output is a **probability distribution** over the vocabulary, summing to 1.